# Training for audio features

In [1]:
from utils import import_data_science, load_music_dataset
from utils import data_transformation
from utils import artifacts_gen
import utils


import_data_science(globals())

In [2]:
raw = np.load('/mnt/g/artifacts/signals.npy', allow_pickle=True)

signals = raw[:,3]
signals.shape


one_hot_encoding = np.vectorize(data_transformation.one_hot_function)


labels = raw[:,1]

labels = one_hot_encoding(labels)
labels.shape

(5000,)

In [3]:
from IPython.display import clear_output

X = []
Y = []

for index, (signal, label) in enumerate(zip(signals, labels)):
    try:
        # print(f"{index / labels.shape[0] * 100}%")
        signal.shape[0] == 308700
        X.append(signal)
        Y.append(label)
        # result.append((features, label))
    except Exception as e:
        # print(f"couldn't extract features, {e}")
        pass
    # clear_output(wait=True)


print(len(X), len(Y))

X_train = []
y_train = []


for (signal, label) in zip(X, Y):
    try:
        # print(f"{index / labels.shape[0] * 100}%")
        for i in range(5,10,1):
            X_train.append(signal[i*22050:(i+1)*22050])
            y_train.append(label)
        # result.append((features, label))
    except Exception as e:
        print(f"couldn't extract features, {e}")
    # clear_output(wait=True)

print(len(X_train), len(y_train))

del X
del Y
del labels
del signals
del raw


4828 4828
24140 24140


In [4]:
def normalize_and_make_numpy(x, y):
    x = np.array(x).astype(np.float32)
    y = np.array(y).astype(np.float32)

    return x, y

x, y = normalize_and_make_numpy(X_train,y_train)

In [5]:
x.shape, y.shape

((24140, 22050), (24140,))

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, TensorDataset, Subset
import torch.nn.functional as F

In [8]:
class AudioDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # shape: (N, 1, 22050)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.01, stratify=y, random_state=42
)


train_dataset = AudioDataset(x_train, y_train)
test_dataset = AudioDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [9]:
len(train_dataset), len(test_dataset)

(23898, 242)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, stride=1, padding=2)
        self.pool1 = nn.MaxPool1d(4)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=1, padding=2)
        self.pool2 = nn.AdaptiveAvgPool1d(1)  

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(32, 5)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))  # (B, 16, ~5512)
        x = self.pool2(F.relu(self.conv2(x)))  # (B, 32, 1)
        x = x.view(x.size(0), -1)              # (B, 32)
        x = self.dropout(x)
        return self.fc(x)

In [11]:
torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    
    print(f"Validation Accuracy: {correct / total:.2%}")

Epoch 1: Loss = 1.4457
Validation Accuracy: 46.28%
Epoch 2: Loss = 1.3443
Validation Accuracy: 49.59%
Epoch 3: Loss = 1.3345
Validation Accuracy: 48.76%
Epoch 4: Loss = 1.3186
Validation Accuracy: 51.65%
Epoch 5: Loss = 1.3138
Validation Accuracy: 45.45%
Epoch 6: Loss = 1.3007
Validation Accuracy: 50.83%
Epoch 7: Loss = 1.2883
Validation Accuracy: 52.89%
Epoch 8: Loss = 1.2779
Validation Accuracy: 49.59%
Epoch 9: Loss = 1.2742
Validation Accuracy: 51.24%
Epoch 10: Loss = 1.2695
Validation Accuracy: 49.59%


In [40]:
model.cpu()
torch.save(model.state_dict(), "/mnt/g/models/audio_cnn_model.pt")

In [13]:

upload_ids = ["001","002","003"]
upload_dir = "/mnt/g/uploads/"
models_dir = "/mnt/g/models/"
artifacts_dir = "/mnt/g/song_artifacts/"
model_filename = "0.pth"
# TODO check and if not present make folders
# songslice
# videos
# power
# mels
# chroma
# mfcc


frame_size = 4096
hop_size = 512
fft_window = 512
signal_duration = 14

In [16]:
# import librosa as li
# import utils

# # Execution
# signal, sr = utils.extract_signal_for_inference(filename)

In [19]:

# # Storing transformed signal
# song_location = utils.generate_audio_from_frames(f"generated_track_{upload_id}", signal, sr, artifacts_dir)

# # Frames indexes for generation of data
# frames = utils.split_signal_to_frame_indexes(frame_size, hop_size, signal)

# def extract_mels_and_generate_artifacts():
#     mels = utils.extract_mels(frames, signal)
#     utils.generate_mel_spec_images(upload_id, mels, artifacts_dir)
#     utils.generate_video_from_mel_spec_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
#     return mels

# def extract_power_spectrograms_and_generate_artifacts():
#     power_spectrograms = utils.extract_power_spectrograms(frames, signal)
#     utils.generate_power_spectr_spec_images(upload_id, power_spectrograms, artifacts_dir)
#     utils.generate_video_from_power_spetr_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
#     return power_spectrograms

# def extract_mfcc_and_generate_artifacts():
#     mfcc = utils.extract_mfccs(frames, signal)
#     utils.generate_mfcc_images(upload_id, mfcc, artifacts_dir)
#     utils.generate_video_from_mfcc_images(upload_id, frame_size, song_location, len(frames), artifacts_dir)
#     return mfcc

# mels = extract_mels_and_generate_artifacts()
# power_spectrograms = extract_power_spectrograms_and_generate_artifacts()
# mfcc = extract_mfcc_and_generate_artifacts()

In [17]:
# stacked = utils.stack_mfcc_power_spectro_mel_spectro(mfcc, mels, power_spectrograms)
# stacked.shape

# x = np.array(stacked).astype(np.float32)

# flattened = torch.from_numpy(x).reshape(-1)

# output = model(flattened)
# percentages = F.softmax(output, dim=0) * 100

In [19]:
# for i in range(5,10,1):
#             X_train.append(signal[i*22050:(i+1)*22050])

In [113]:
# import IPython.display as ipd



# frames = []
# for index in range(1, 14, 1):
#     frames.append(signal[index*22050:(index+1)*22050])
#     # ipd.Audio(data= signal[index*22050:(index+1)*22050], rate=22050)

# ipd.Audio(data=frames[1], rate=22050)
    

In [20]:
# ipd.Audio(data=test_dataset.X[0][0], rate=22050)
    


In [38]:
model.eval()
from torch import Tensor
from collections import Counter

for upload_id in upload_ids:
    print(f"infering from {upload_id}") 
    
    filename = utils.find_file_by_upload_id(upload_id, upload_dir)
    model.eval()
    # Execution
    signal, sr = utils.extract_signal_for_inference(filename)

    
    X = []
    preds = {}
    class_guesses = []
    
    for i in range(1,14,1):
        X.append(signal[i*22050:(i+1)*22050])
    
    for index, frame in enumerate(X):
        waveform = torch.from_numpy(frame).float().unsqueeze(0).unsqueeze(0)  # shape (1, 1, 22050)
        waveform = waveform.to(device)
        with torch.no_grad():
            output = model(waveform)
            prediction = F.softmax(output, dim=1) * 100
            
            preds[index] = (prediction.cpu().numpy())
            class_id = torch.argmax(output)
            class_guesses.append(utils.reverse_one_hot(class_id.item())) 

    mean_pred = np.zeros_like(next(iter(preds.values())))
    for pred in preds.values():
        mean_pred += pred
    mean_pred /= len(preds)

    all_preds = np.stack(list(preds.values()), axis=0).squeeze(1)
    median_pred = np.median(all_preds, axis=0) 
    
    counter = Counter(class_guesses)
    print(f"preds: {preds}\n\n, mean_pred: {mean_pred}\n\nprediction_medians: {median_pred}\n\n, class guesses: {counter}\n\n------")
# Metallica : Predicted class: tensor([[ 9.3146,  8.1708, 22.2267, 36.9664, 23.3215]], device='cuda:0')

# Predicted class: tensor([[22.6379, 16.7982, 19.2500, 18.4610, 22.8529]], device='cuda:0') # 5 epochs metallica

infering from 001
Upload couldn't be found
Upload couldn't be found
preds: {0: array([[9.3169651e+00, 3.6333187e+01, 2.5539860e+01, 2.8809660e+01,
        3.2998441e-04]], dtype=float32), 1: array([[37.723343 , 15.697287 , 13.561996 , 27.768278 ,  5.2490973]],
      dtype=float32), 2: array([[34.168064 , 17.45774  , 16.573568 , 27.577082 ,  4.2235384]],
      dtype=float32), 3: array([[22.006083  , 26.332733  , 23.797304  , 27.756523  ,  0.10735929]],
      dtype=float32), 4: array([[30.20451  , 20.902735 , 19.835348 , 27.768484 ,  1.2889239]],
      dtype=float32), 5: array([[33.752308 , 18.175383 , 17.543634 , 28.035563 ,  2.4931166]],
      dtype=float32), 6: array([[34.722404 , 16.486519 , 14.325608 , 27.235096 ,  7.2303667]],
      dtype=float32), 7: array([[30.3651   , 20.819124 , 19.274715 , 27.271887 ,  2.2691748]],
      dtype=float32), 8: array([[29.44133  , 21.277258 , 20.469776 , 27.01982  ,  1.7918153]],
      dtype=float32), 9: array([[31.372082 , 20.110594 , 18.025293 , 